In [1]:
from ase.data import atomic_masses
import numpy as np

def structure_density(atoms):

    masses = atoms.get_masses().sum()  # amu

    volume = atoms.get_volume()        # Å^3

    # convert amu/Å^3 → g/cm^3
    density = masses * 1.66054 / volume

    return density
from ase.build import bulk
import numpy as np
from ase.io import write
from ase.visualize import view
a_eq = 5.41

# Build clean bulk CeO2
atoms_eq = bulk('CeO2', 'fluorite', a=a_eq)
atoms_eq.set_pbc([True, True, True])


#ceria_sto=atoms_eq*[4,4,4]

ref_dens=structure_density(atoms_eq)
print(ref_dens)

7.219942444386959


# Functions (criteria)

In [2]:
def is_slab(atoms, vacuum_threshold=8.0):

    a, b, c = atoms.cell.lengths()
    lengths = np.array([a, b, c])

    # identify vacuum axis
    max_len = lengths.max()
    min_len = lengths.min()

    if max_len - min_len > vacuum_threshold:
        return True

    return False

In [3]:
def is_low_density(atoms, ref_dens, cutoff=0.7):
    dens = structure_density(atoms)
    return dens < cutoff * ref_dens

In [4]:
def chemistry_class(atoms):
    symbols = atoms.get_chemical_symbols()

    # Allow only Ce, O, and Pr
    allowed = {"Ce", "O", "Pr"}

    if not set(symbols).issubset(allowed):
        return "other"

    n_pr = symbols.count("Pr")

    if n_pr == 0:
        return "stoichiometric"

    return "redox"

## Filtering

In [5]:
import numpy as np
from ase.db import connect
db_in=connect("no_adsorbates_ceria.db")
db_out_redox= connect("redox_09-06.db")
db_out_bulk=connect("sto_09-06.db")
db_out_sto_slab=connect("sto_slab_09-06.db")


In [6]:
kept_bulk=0
kept_sto_slab=0
for row in db_in.select():
    atoms = row.toatoms()

    if chemistry_class(atoms) != "stoichiometric":
        continue

    # STO bulk: density matches bulk reference
    if not is_low_density(atoms, ref_dens):
        db_out_bulk.write(atoms, data=row.data)
        kept_bulk += 1

    # STO slab: density lower than bulk and vacuum in one direction
    elif is_slab(atoms):
        db_out_sto_slab.write(atoms, data=row.data)
        kept_sto_slab += 1
#print("STO bulk:", kept_bulk, "STO slab:", 
kept_sto_slab)

STO bulk: 1 STO slab: 1
STO bulk: 6 STO slab: 2
STO bulk: 6 STO slab: 3
STO bulk: 6 STO slab: 4
STO bulk: 6 STO slab: 5
STO bulk: 6 STO slab: 6
STO bulk: 6 STO slab: 7
STO bulk: 6 STO slab: 8
STO bulk: 6 STO slab: 9
STO bulk: 6 STO slab: 10
STO bulk: 6 STO slab: 11
STO bulk: 7 STO slab: 12
STO bulk: 7 STO slab: 13
STO bulk: 23 STO slab: 14
STO bulk: 23 STO slab: 15
STO bulk: 23 STO slab: 16
STO bulk: 23 STO slab: 17
STO bulk: 23 STO slab: 18
STO bulk: 23 STO slab: 19
STO bulk: 23 STO slab: 20
STO bulk: 23 STO slab: 21
STO bulk: 23 STO slab: 22
STO bulk: 23 STO slab: 23
STO bulk: 23 STO slab: 24
STO bulk: 23 STO slab: 25
STO bulk: 23 STO slab: 26
STO bulk: 23 STO slab: 27
STO bulk: 23 STO slab: 28
STO bulk: 23 STO slab: 29
STO bulk: 23 STO slab: 30
STO bulk: 23 STO slab: 31
STO bulk: 23 STO slab: 32
STO bulk: 23 STO slab: 33
STO bulk: 23 STO slab: 34
STO bulk: 23 STO slab: 35
STO bulk: 23 STO slab: 36
STO bulk: 23 STO slab: 37
STO bulk: 23 STO slab: 38
STO bulk: 23 STO slab: 39
STO bulk

In [9]:
print(" Bulk:", len(db_out_bulk), "Slabs:", len(db_out_sto_slab))

 Bulk: 23 Slabs: 112
